# PaySim SQL Exploration

This notebook is the first analysis artifact for `paysim-deep-dive`. The goal is to answer the first five questions a senior analyst should ask after receiving a new payments dataset:

1. What transaction types exist, and where does fraud concentrate?
2. How large are transactions overall and by type?
3. Which originators transact most often?
4. Which destinations receive the most inflow?
5. When does fraud spike across PaySim's simulated time steps?

The notebook uses DuckDB so the 6.3M-row CSV can be queried directly without loading the whole file into pandas first.


## Setup

Expected raw data path: `data/PS_20174392719_1491204439457_log.csv`.

If the file is missing, run from the repo root:

```bash
pip install -r requirements.txt
kaggle datasets download -d ealaxi/paysim1 --unzip -p data/
```


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

candidate_paths = [
    Path("data/PS_20174392719_1491204439457_log.csv"),
    Path("../data/PS_20174392719_1491204439457_log.csv"),
    Path("data/paysim.csv"),
    Path("../data/paysim.csv"),
]
DATA_PATH = next((path for path in candidate_paths if path.exists()), candidate_paths[0])
DATA_PATH


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"PaySim CSV not found at {DATA_PATH}. "
        "Download it with: kaggle datasets download -d ealaxi/paysim1 --unzip -p data/"
    )

con = duckdb.connect()
con.execute("CREATE OR REPLACE VIEW paysim AS SELECT * FROM read_csv_auto(?)", [str(DATA_PATH)])


## Dataset Sanity Check

Before analysis, confirm row count, time range, transaction types, and fraud prevalence. This protects the rest of the notebook from silently analyzing the wrong file.


In [ ]:
sanity_query = """
SELECT
    COUNT(*) AS rows,
    MIN(step) AS min_step,
    MAX(step) AS max_step,
    COUNT(DISTINCT type) AS transaction_types,
    SUM(isFraud) AS fraud_rows,
    AVG(isFraud) AS fraud_rate
FROM paysim;
"""

sanity = con.sql(sanity_query).df()
sanity


## Query 1 — Transaction Mix And Fraud Rate By Type

Question: which transaction types dominate volume, and where does fraud actually appear?


In [ ]:
query_1 = """
SELECT
    type,
    COUNT(*) AS transactions,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS transaction_share_pct,
    SUM(isFraud) AS fraud_transactions,
    ROUND(100.0 * AVG(isFraud), 4) AS fraud_rate_pct,
    SUM(isFlaggedFraud) AS flagged_transactions
FROM paysim
GROUP BY type
ORDER BY transactions DESC;
"""

q1_type_fraud = con.sql(query_1).df()
q1_type_fraud


**Finding stub:** Fraud should be concentrated in `TRANSFER` and `CASH_OUT`; if other types show fraud, investigate schema or import issues.


## Query 2 — Amount Distribution By Type

Question: do transaction amounts differ enough by type to matter for downstream analysis?


In [ ]:
query_2 = """
SELECT
    type,
    COUNT(*) AS transactions,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(MEDIAN(amount), 2) AS median_amount,
    ROUND(QUANTILE_CONT(amount, 0.90), 2) AS p90_amount,
    ROUND(QUANTILE_CONT(amount, 0.99), 2) AS p99_amount,
    ROUND(MAX(amount), 2) AS max_amount
FROM paysim
GROUP BY type
ORDER BY median_amount DESC;
"""

q2_amount_by_type = con.sql(query_2).df()
q2_amount_by_type


**Finding stub:** Compare median vs p99 to describe skew. This is more useful than reporting only the mean.


## Query 3 — Most Active Originators

Question: are originator IDs mostly one-off accounts, or do repeated originators exist?


In [ ]:
query_3 = """
SELECT
    nameOrig,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT type) AS distinct_transaction_types,
    ROUND(SUM(amount), 2) AS total_sent,
    SUM(isFraud) AS fraud_transactions,
    MIN(step) AS first_step,
    MAX(step) AS last_step
FROM paysim
GROUP BY nameOrig
ORDER BY transaction_count DESC, total_sent DESC
LIMIT 20;
"""

q3_top_originators = con.sql(query_3).df()
q3_top_originators


**Finding stub:** PaySim originators are expected to be near-unique. Repeated originators may still reveal simulator behavior worth noting.


## Query 4 — Top Destinations By Inflow

Question: which destination accounts receive the most value, and are they customers (`C...`) or merchants (`M...`)?


In [ ]:
query_4 = """
SELECT
    nameDest,
    CASE
        WHEN STARTS_WITH(nameDest, 'M') THEN 'merchant'
        WHEN STARTS_WITH(nameDest, 'C') THEN 'customer'
        ELSE 'unknown'
    END AS dest_kind,
    COUNT(*) AS incoming_transactions,
    ROUND(SUM(amount), 2) AS total_inflow,
    ROUND(AVG(amount), 2) AS avg_inflow,
    SUM(isFraud) AS fraud_transactions,
    MIN(step) AS first_step,
    MAX(step) AS last_step
FROM paysim
GROUP BY nameDest, dest_kind
ORDER BY total_inflow DESC
LIMIT 20;
"""

q4_top_destinations = con.sql(query_4).df()
q4_top_destinations


**Finding stub:** Use this to discuss receiver concentration and why merchant/customer prefixes matter for interpretation.


## Query 5 — Fraud Over Simulated Time

Question: does fraud appear evenly across PaySim steps, or are there spikes worth investigating?


In [ ]:
query_5 = """
WITH by_step AS (
    SELECT
        step,
        COUNT(*) AS transactions,
        SUM(isFraud) AS fraud_transactions,
        AVG(isFraud) AS fraud_rate
    FROM paysim
    GROUP BY step
)
SELECT
    step,
    transactions,
    fraud_transactions,
    ROUND(100.0 * fraud_rate, 4) AS fraud_rate_pct
FROM by_step
WHERE fraud_transactions > 0
ORDER BY fraud_transactions DESC, fraud_rate DESC
LIMIT 25;
"""

q5_fraud_by_step = con.sql(query_5).df()
q5_fraud_by_step


**Finding stub:** Separate absolute fraud spikes from high-rate low-volume steps. Both can be true, but they imply different operational responses.


## README Findings Draft

After running the notebook, copy 3-5 bullets into `README.md` under a `## Initial findings` section.

Candidate bullets:

- Fraud is not spread across all transaction types; it concentrates in the transaction types that can drain value.
- Transaction amounts are heavily skewed, so medians and tail percentiles are more informative than averages alone.
- Originator behavior is sparse; destination behavior has more repeated entities and is better for concentration analysis.
- Time-step spikes need both count and rate views to avoid overreacting to tiny denominators.
- `isFlaggedFraud` is a very narrow rule baseline, not a full fraud-detection strategy.


## Next Steps

- Add 3-5 findings to `README.md`.
- Add one small chart in a Python EDA notebook only after the SQL notebook is committed.
- Keep modeling out of scope until the EDA story is readable.
